# Analysis for the cosmology benchmark (C functions)

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
x1, x2 = sympy.symbols("x1:3")
x = sympy.abc.x
y = sympy.abc.y

## Loading data

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report.run_set.unique()

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
f_indices = full_report.data_set.apply(lambda x: x.startswith("C"))
fr2 = full_report.loc[f_indices].set_index(["run_set", "data_set", "sample_num"])

In [ ]:
fr2["sympy"] = fr2.expr_original_syms.apply(au.parse_if_needed)
fr2["sympy_defuzz"] = fr2.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
srb = fr2.loc["SRB-2026-06-26-1130-arr8"]
cht = fr2.loc["CHT-2026-06-28-2315"]

In [ ]:
srb.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
min_mse_ixs = srb.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb.loc[min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht_min_mse_ixs = cht.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

## Polynomials

In [ ]:
data_sets_polynomial = ["C2a"]

In [ ]:
srb.loc[data_sets_polynomial]

`C2a` is no problem.

I used to include `C5f` here, but it's a rational function, not a polynomial.
Table on p31 of the cosmology article is confusing, because `C2a` has a reference to $H(z)$ but there's a column of $H$ in the data file and it really is a simple polynomial.
But `C5f`, which looks like the same kind of item, seems to be referring to `C5d` and `C5e` as functions or $R_0$ and $r$.

In [ ]:
srb.loc[min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
(srb.loc[data_sets_polynomial]
 .sympy_defuzz.apply(lambda e: not e.is_polynomial(x1, x2))
 .groupby(level="data_set")
 .sum())

All runs on all polynomial data sets are correct up to fuzz.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_polynomial])

In [ ]:
sns.displot(data=srb.loc[data_sets_polynomial],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=srb.loc[data_sets_polynomial],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

## TODO Rational functions

In [ ]:
data_sets_rational = [
    "d_bacres1",
    "d_bacres2",
    "d_predprey1",
    "d_predprey2"
    ]

In [ ]:
srb.loc[data_sets_rational]

In [ ]:
srb.loc[min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Using the CHT configuration, all of these are correct with some additional defuzzing and simplification.

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht_bacres1 = cht.loc[cht_min_mse_ixs].loc["d_bacres1", "sympy"].iloc[0]
cht_bacres1

In [ ]:
au.replace_near_integer(sympy.expand(cht_bacres1), tolerance=5e-3)

In [ ]:
cht_bacres2 = cht.loc[cht_min_mse_ixs].loc["d_bacres2", "sympy"].iloc[0]
cht_bacres2

In [ ]:
au.replace_near_integer(sympy.expand(cht_bacres2), tolerance=5e-3)

Using the SRB configuration, the bacterial respiration ones are not quite right symbolically.
`bacres1` has some imperfection in a denominator.

In [ ]:
srb_bacres1 = srb.loc[min_mse_ixs].loc["d_bacres1", "sympy"].iloc[0]
srb_bacres1

In [ ]:
au.replace_near_integer(srb_bacres1, tolerance=5e-5)

`bacres2` is actually correct, it's just hard to simplify it down all the way.

In [ ]:
srb_bacres2 = srb.loc[min_mse_ixs].loc["d_bacres2", "sympy"].iloc[0]
srb_bacres2

In [ ]:
sympy.expand(srb_bacres2)

In [ ]:
au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5)

In [ ]:
sympy.nsimplify(au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5), tolerance=4e-2)

In [ ]:
sympy.simplify(sympy.nsimplify(au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5), tolerance=4e-2))

The majority of solutions are not rational functions.

In [ ]:
(srb.loc[data_sets_rational]
 .sympy_defuzz
 .apply(lambda e: not e.is_rational_function())
 .groupby(level="data_set").sum())

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational])

In [ ]:
bacres1_rf_ixs = srb.loc["d_bacres1"].sympy_defuzz.apply(lambda e: e.is_rational_function())

In [ ]:
srb.loc["d_bacres1"][bacres1_rf_ixs].sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

If we cheat, all of them have to be rational functions, and more are correct.

In [ ]:
au.count_by_threshold(cht.loc[data_sets_rational])

In [ ]:
cht.loc["d_bacres1"].sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

In [ ]:
sns.stripplot(
    data=srb.loc[data_sets_rational],
    x="complexity",
    y="mse",
    hue="data_set",
    log_scale=[False,True])

In [ ]:
sns.displot(data=srb.loc[data_sets_rational],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=srb.loc[data_sets_rational],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

In [ ]:
sns.displot(data=cht.loc[data_sets_rational],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )